# Lecture 18 — Energy-Based Models: Score Matching to Partition Functions

**PHYG004 / PHY5006, 2026 Spring · Sogang University**
Prof. Young Woo Choi

---

## Where we are in the course

This is the last stop in our **generative-models block**:

| Lecture | Model | Core object |
|---|---|---|
| L14 | VAE | encoder + decoder, ELBO |
| L15 | Normalizing Flow | invertible $f$, exact $\log p$ |
| L16/L17 | Diffusion / DDPM | the **score** $\nabla_x \log p_t(x)$ |
| **L18 (today)** | **Energy-Based Model** | the **energy** $E_\theta(x)$ and its gradient |

The key sentence for a physicist:

> **An energy-based model is just a Boltzmann distribution whose energy function is a neural network.**

Everything you already know about statistical mechanics — partition functions, free energies, Langevin dynamics, slow mixing across barriers — transfers *directly*. Today we make that bridge explicit and build the full pipeline from scratch in JAX:

$$
\text{learn } E_\theta(x) \;\xrightarrow{\;\text{denoising score matching}\;}\; \text{score } s_\theta = -\nabla E_\theta \;\xrightarrow{\;\text{Langevin MCMC}\;}\; \text{samples}.
$$

## What you'll be able to do by the end

1. Write down the EBM / Boltzmann dictionary: $E(x) \sim -\log p(x)$, $Z(T) \sim$ normalizing constant, Langevin $\sim$ overdamped Brownian motion.
2. Explain **why $Z$ is intractable** and why that forces us away from maximum likelihood.
3. Implement **denoising score matching (DSM)** and understand *why* it secretly learns an energy without ever touching $Z$.
4. See, in equations and in code, that the **DDPM score from L17** and the **$-\nabla E/kT$ of an EBM** are *the same mathematical object*.
5. Sample an EBM with **Langevin MCMC** and diagnose its **slow mixing** across energy barriers — a genuinely physical failure mode.
6. Train a 2D EBM on the **alanine-dipeptide Ramachandran $(\phi,\psi)$ free-energy surface** — the same molecule we sampled in L15–L17.

> **Runtime.** The 2D toy and the alanine-dipeptide 2D EBM both train on the free Colab **CPU** in a few minutes. Only the optional MD-trajectory extraction needs `mdtraj` + a download.


## 0. Setup

The first cell installs everything (uncomment on Colab). The second cell does imports, fixes the PRNG seed, and reports the backend.

In [ ]:
# On Google Colab, uncomment the next line to install dependencies.
# !pip install -q jax jaxlib optax flax matplotlib
#
# Optional (only for the real alanine-dipeptide MD section, §9):
# !pip install -q mdtraj


In [ ]:
import jax
import jax.numpy as jnp
import jax.random as jr
import optax
import flax.nnx as nnx
import numpy as np
import matplotlib.pyplot as plt
from functools import partial

# Reproducibility — every PRNG key in this notebook descends from this one.
SEED = 42
key = jr.PRNGKey(SEED)
np.random.seed(SEED)

print(f"JAX version : {jax.__version__}")
print(f"JAX backend : {jax.default_backend()}")


## 1. The physicist's dictionary: energy ⇄ probability

Start from statistical mechanics. A system with energy function $U(x)$ at temperature $T$ sits in the **Boltzmann distribution**

$$
p(x) \;=\; \frac{1}{Z}\, e^{-U(x)/k_B T}, \qquad
Z \;=\; \int e^{-U(x)/k_B T}\, dx .
$$

Take logs and rearrange:

$$
U(x) \;=\; -k_B T \,\log p(x) \;-\; k_B T \log Z .
$$

So **energy is just negative log-probability**, up to an additive constant. An *energy-based model* turns this around: pick a flexible function $E_\theta(x)$ (a neural net) and *define*

$$
\boxed{\,p_\theta(x) = \frac{1}{Z_\theta}\, e^{-E_\theta(x)}\,}, \qquad Z_\theta = \int e^{-E_\theta(x)}\, dx .
$$

We have absorbed $k_B T$ into $E_\theta$ (work in units of $k_BT$; we restore $k_BT = 0.6$ kcal/mol at 300 K when we need physical numbers). The dictionary every term in this notebook obeys:

| Machine learning | Statistical physics |
|---|---|
| energy $E_\theta(x)$ | dimensionless potential $U(x)/k_BT$ |
| $-\log p_\theta(x)$ | energy + const |
| normalizing constant $Z_\theta$ | **partition function** $Z(T)$ |
| $-\log Z_\theta$ | free energy $F/k_BT$ |
| score $s_\theta(x)=\nabla_x \log p_\theta(x)$ | **force** $-\nabla U /k_BT$ |
| Langevin sampler step | overdamped **Brownian dynamics** |
| sampling across modes | barrier crossing / slow mixing |

This table *is* the lecture. The single most important line:

$$
\boxed{\,s_\theta(x) \;\equiv\; \nabla_x \log p_\theta(x) \;=\; -\nabla_x E_\theta(x)\,}
$$

Notice the **$Z_\theta$ dropped out** — it is a constant in $x$, so it vanishes under $\nabla_x$. This is the whole reason EBMs are trainable at all. We will never compute $Z_\theta$.


### 1.1 The one hard fact: $Z_\theta$ is intractable

For $x \in \mathbb{R}^d$, $Z_\theta = \int e^{-E_\theta(x)}\,dx$ is a $d$-dimensional integral with no closed form. For alanine dipeptide's full Cartesian coordinates that's a $\sim$66-dimensional integral; for an image it's a million-dimensional one. We **cannot** evaluate $p_\theta(x)$, so we **cannot** do ordinary maximum likelihood.

Two roads around this — both are physics-flavored:

- **Score matching** (today's main road): match $\nabla_x \log p_\theta = -\nabla_x E_\theta$ to the data's score. $Z_\theta$ never appears because $\nabla_x \log Z_\theta = 0$.
- **Contrastive divergence / MCMC-in-the-loop**: estimate the gradient of $\log Z_\theta$ with samples. We sketch it briefly but do not use it.

Let's first *see* a partition function with our own eyes in 1D, where we *can* integrate.

In [ ]:
# A 1D double-well energy, in units of kT. Two minima => bistable system.
def U_double_well(x, a=1.0, b=0.0):
    # E(x) = a (x^2 - 1)^2 + b x  : minima near x = +/-1, optional tilt b
    return a * (x**2 - 1.0)**2 + b * x

xs = jnp.linspace(-2.5, 2.5, 600)
E = U_double_well(xs)
unnorm = jnp.exp(-E)                       # e^{-E(x)}  (un-normalized Boltzmann weight)
Z = jnp.trapezoid(unnorm, xs)              # partition function by numerical quadrature
p = unnorm / Z                             # normalized density

print(f"xs shape       : {xs.shape}")
print(f"E(x) shape     : {E.shape}")
print(f"Z (1D integral): {float(Z):.4f}")
print(f"free energy -logZ: {float(-jnp.log(Z)):.4f}")

fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
ax[0].plot(xs, E, lw=2); ax[0].set_title("energy  E(x)"); ax[0].set_xlabel("x"); ax[0].set_ylabel("E / kT")
ax[1].plot(xs, p, lw=2, color="C3"); ax[1].set_title(r"$p(x)=e^{-E(x)}/Z$"); ax[1].set_xlabel("x")
for a_ in ax: a_.grid(alpha=0.3)
plt.tight_layout(); plt.show()


**Read this off the plot.** The energy has two valleys at $x=\pm1$ separated by a barrier at $x=0$. The probability has two peaks there. In 1D we computed $Z$ by `trapezoid` in one line. In $d=66$ that quadrature has $600^{66}$ grid points — utterly impossible. *That* is the wall every EBM method is built to avoid.

## 2. Bridge from L17: the score you already know

In **L16/L17 (diffusion)** you trained a network $\epsilon_\theta(x_t, t)$ to predict the noise added to a sample. The central identity there was

$$
s_\theta(x_t, t) \;=\; \nabla_{x_t} \log p_t(x_t) \;=\; -\,\frac{\epsilon_\theta(x_t,t)}{\sigma_t}.
$$

DDPM's $\epsilon$-prediction **is** score regression, scaled by the noise level. Today's EBM produces the **same object** by a different route:

$$
\underbrace{s_\theta(x) = -\nabla_x E_\theta(x)}_{\text{L18 EBM}}
\qquad\Longleftrightarrow\qquad
\underbrace{s_\theta(x_t,t) = -\epsilon_\theta/\sigma_t}_{\text{L17 diffusion}} .
$$

The difference is only in **parameterization**:

| | what the net outputs | score obtained by | gives you an explicit $E$? |
|---|---|---|---|
| **DDPM (L17)** | a *vector* $\epsilon_\theta(x,t)$ | $s = -\epsilon/\sigma_t$ | no — only the score |
| **EBM (L18)** | a *scalar* $E_\theta(x)$ | $s = -\nabla_x E_\theta$ (autograd) | **yes** — the energy itself |

Because the EBM outputs a **scalar**, its score field is the gradient of a potential and is therefore **automatically conservative** (curl-free): $\nabla\times s_\theta = 0$ by construction. A free-form vector score $\epsilon_\theta$ has no such guarantee. For physics — where forces *are* gradients of a potential — the EBM parameterization is the physically honest one. That is today's punchline, and we will verify it numerically in §8.

## 3. The energy network $E_\theta$  ·  ⭐ Exercise (TODO 1)

We need a map $x \in \mathbb{R}^{B\times d} \mapsto E_\theta(x) \in \mathbb{R}^{B}$ — a neural network with a **scalar** output. We use `flax.nnx` (the same API as L14–L17).

**Design choices that matter for an energy:**

- The output dimension of the **last** `Linear` must be **1**, then we `squeeze` to shape `(B,)`.
- We use a **smooth** activation (`tanh` / `softplus`), *not* `relu`. We will differentiate $E_\theta$ twice-ish (score = first derivative, and the optimizer pushes on it), so kinks from `relu` hurt. Smoothness ↔ a well-defined force field.

> ### ⭐ TODO 1 — complete `EnergyNet.__call__`
> Fill in the forward pass so that `E(x).shape == (B,)`.
> **Verification:** on the double-well, the energy at a valley $(\pm1)$ must be *lower* than at the ridge $(0)$ once trained — and even at init the shapes must be right.

In [ ]:
class EnergyNet(nnx.Module):
    '''Scalar energy E_theta : (B, d) -> (B,).'''

    def __init__(self, d_in: int, hidden: int = 128, *, rngs: nnx.Rngs):
        self.lin1 = nnx.Linear(d_in, hidden, rngs=rngs)
        self.lin2 = nnx.Linear(hidden, hidden, rngs=rngs)
        self.lin3 = nnx.Linear(hidden, hidden, rngs=rngs)
        # NOTE: scalar output -> out_features = 1
        self.out = nnx.Linear(hidden, 1, rngs=rngs)

    def __call__(self, x: jax.Array) -> jax.Array:
        # x: (B, d) -> E: (B,)
        # ---------------------------------------------------------------
        # TODO 1: implement the forward pass.
        #   - apply lin1, lin2, lin3 each followed by a smooth activation
        #     (nnx.tanh or jax.nn.softplus), then self.out (-> shape (B, 1)),
        #   - finally squeeze the last axis so the result is (B,).
        # Replace the next line.
        h = nnx.tanh(self.lin1(x))
        h = nnx.tanh(self.lin2(h))
        h = nnx.tanh(self.lin3(h))
        e = self.out(h)                # (B, 1)
        return jnp.squeeze(e, axis=-1) # (B,)


In [ ]:
# --- shape + sanity check for TODO 1 ---
enet = EnergyNet(d_in=1, hidden=64, rngs=nnx.Rngs(0))

x_valley = jnp.array([[-1.0], [1.0]])   # the two double-well minima
x_ridge  = jnp.array([[0.0]])           # the barrier top
E_valley = enet(x_valley)
E_ridge  = enet(x_ridge)

print(f"E(valley) shape : {E_valley.shape}   (expected (2,))")
print(f"E(ridge)  shape : {E_ridge.shape}    (expected (1,))")
assert E_valley.shape == (2,), "TODO 1: output must be (B,)"
print("shape check passed. (At init E is random; the energy ordering")
print(" E_valley < E_ridge becomes meaningful only AFTER training in section 5.)")


## 4. The score is one `jax.grad` away

The score field is, by definition, $s_\theta(x) = -\nabla_x E_\theta(x)$. In JAX this is literally one line. We sum over the batch so that `jax.grad` returns the per-sample gradient (a standard JAX trick — the cross terms are zero because $E_\theta$ acts independently per row).

In [ ]:
def energy_sum(model, x):
    # scalar = sum_b E_theta(x_b); its grad wrt x is the stacked per-sample grad
    return jnp.sum(model(x))

def score_fn(model, x):
    '''s_theta(x) = -grad_x E_theta(x), shape (B, d).'''
    g = jax.grad(energy_sum, argnums=1)(model, x)
    return -g

x_test = jnp.array([[-1.0], [0.0], [1.0]])
s = score_fn(enet, x_test)
print(f"x       shape: {x_test.shape}")
print(f"score   shape: {s.shape}   (must match x: same dim as data)")
print(f"score values : {np.array(s).ravel()}  (random at init)")


## 5. Training without $Z$: Denoising Score Matching  ·  ⭐ Exercise (TODO 2)

We cannot maximize likelihood (no $Z_\theta$). Instead we match scores. The cleanest, most stable version is **Denoising Score Matching** (Vincent 2011) — which, not coincidentally, is *exactly* what L17's diffusion loss was.

**The recipe.** Pick a noise scale $\sigma$. Corrupt each data point:
$$
\tilde x = x + \sigma\,\epsilon, \qquad \epsilon \sim \mathcal N(0, I).
$$
The conditional kernel is a Gaussian $p_\sigma(\tilde x \mid x) = \mathcal N(\tilde x; x, \sigma^2 I)$, whose score is known in closed form:
$$
\nabla_{\tilde x}\log p_\sigma(\tilde x \mid x) \;=\; -\frac{\tilde x - x}{\sigma^2} \;=\; -\frac{\epsilon}{\sigma}.
$$
Vincent's theorem says the model score $s_\theta(\tilde x)$ that minimizes
$$
\boxed{\;\mathcal L_{\text{DSM}}(\theta) \;=\; \mathbb E_{x,\epsilon}\Big\|\, s_\theta(\tilde x) \;-\; \big(-\tfrac{\epsilon}{\sigma}\big)\,\Big\|^2 \;}
$$
equals the score of the **smoothed data density** $p_\sigma(\tilde x)$. Push $\sigma \to 0$ and you recover the score of the data itself. **$Z_\theta$ never appears.** (Careful: in the literature you'll also see the target written $-(\tilde x - x)/\sigma^2 = -\epsilon/\sigma^2 \cdot \sigma$. We use the $-\epsilon/\sigma$ form, consistent with the kernel score above.)

This is the same loss as DDPM: there the network output was $\epsilon_\theta$, here it is $s_\theta = -\nabla E_\theta$. Same regression, different head.

> ### ⭐ TODO 2 — complete the DSM loss
> Given a clean batch `x`, sample $\epsilon$, build $\tilde x = x + \sigma\epsilon$, compute $s_\theta(\tilde x)$, and return the mean squared error to the target $-\epsilon/\sigma$.
> **Verification:** the loss is **positive** and **decreases** during training (we plot it).

In [ ]:
def dsm_loss(model, x, key, sigma=0.1):
    '''Denoising score matching loss.  x: (B, d) -> scalar.'''
    eps = jr.normal(key, x.shape)             # (B, d)
    # ---------------------------------------------------------------
    # TODO 2:
    #   1. x_tilde = x + sigma * eps
    #   2. s = score_fn(model, x_tilde)        # model score at the noised point
    #   3. target = -eps / sigma               # closed-form kernel score
    #   4. return mean over batch of ||s - target||^2
    x_tilde = x + sigma * eps
    s = score_fn(model, x_tilde)
    target = -eps / sigma
    return jnp.mean(jnp.sum((s - target) ** 2, axis=-1))

# quick smoke test: loss must be a positive scalar
enet_tmp = EnergyNet(d_in=1, hidden=64, rngs=nnx.Rngs(1))
xb = jr.normal(jr.PRNGKey(7), (256, 1))
L0 = dsm_loss(enet_tmp, xb, jr.PRNGKey(8), sigma=0.1)
print(f"DSM loss (untrained): {float(L0):.3f}   (positive scalar — good)")
assert L0 > 0


### 5.1 Train the EBM on the 1D double-well

Data: exact Boltzmann samples from the double-well, drawn by inverse-CDF (cheap in 1D). Then we run plain SGD on the DSM loss.

In [ ]:
# Exact 1D Boltzmann samples via inverse-CDF sampling of p(x) = e^{-U}/Z.
def boltzmann_samples_1d(n, key, a=1.0, b=0.0, lo=-2.5, hi=2.5, ngrid=2000):
    grid = jnp.linspace(lo, hi, ngrid)
    w = jnp.exp(-U_double_well(grid, a, b))
    cdf = jnp.cumsum(w); cdf = cdf / cdf[-1]
    u = jr.uniform(key, (n,))
    idx = jnp.searchsorted(cdf, u)
    return grid[jnp.clip(idx, 0, ngrid - 1)][:, None]   # (n, 1)

data_key, key = jr.split(key)
X = boltzmann_samples_1d(4000, data_key)        # (4000, 1)
print(f"training data shape: {X.shape}")

# fresh model + optimizer
model = EnergyNet(d_in=1, hidden=128, rngs=nnx.Rngs(0))
opt = nnx.Optimizer(model, optax.adam(2e-3), wrt=nnx.Param)

@nnx.jit
def train_step(model, opt, xb, key, sigma):
    loss, grads = nnx.value_and_grad(dsm_loss)(model, xb, key, sigma)
    opt.update(model, grads)
    return loss

SIGMA = 0.10
steps, batch = 1500, 512
loss_hist = []
tkey = jr.PRNGKey(123)
for it in range(steps):
    tkey, bk, nk = jr.split(tkey, 3)
    idx = jr.randint(bk, (batch,), 0, X.shape[0])
    loss = train_step(model, opt, X[idx], nk, SIGMA)
    loss_hist.append(float(loss))
    if it % 250 == 0:
        print(f"step {it:4d}   DSM loss = {float(loss):.4f}")

plt.figure(figsize=(5, 3))
plt.plot(loss_hist); plt.yscale("log")
plt.xlabel("step"); plt.ylabel("DSM loss"); plt.title("loss decreases (TODO 2 check)")
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()


### 5.2 Did it learn the energy? (TODO 1 verification, completed)

We never told the model the analytic energy — only noisy samples. The learned $E_\theta$ is defined **up to an additive constant** (remember $-\log Z$), so we shift both curves to a common minimum before comparing. We also check the *gradient* (the force), which is the physically meaningful, constant-free quantity.

In [ ]:
xs_ = jnp.linspace(-2.2, 2.2, 400)[:, None]
E_true = U_double_well(xs_[:, 0])
E_learn = model(xs_)
# align additive constant (energies are defined up to -log Z)
E_true_s  = E_true  - E_true.min()
E_learn_s = E_learn - E_learn.min()

# scores (forces): -dE/dx, constant-free
s_true = -jax.grad(lambda z: jnp.sum(U_double_well(z)))(xs_[:, 0])
s_learn = score_fn(model, xs_)[:, 0]

fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
ax[0].plot(xs_[:,0], E_true_s, label="true U(x)", lw=2)
ax[0].plot(xs_[:,0], E_learn_s, "--", label=r"learned $E_\theta$", lw=2)
ax[0].set_title("energy (shifted to min 0)"); ax[0].legend(); ax[0].set_xlabel("x")
ax[1].plot(xs_[:,0], s_true, label="true force", lw=2)
ax[1].plot(xs_[:,0], s_learn, "--", label=r"$-\nabla E_\theta$", lw=2)
ax[1].set_title("score / force"); ax[1].legend(); ax[1].set_xlabel("x")
for a_ in ax: a_.grid(alpha=0.3)
plt.tight_layout(); plt.show()

# TODO 1 verification, post-training: valley energy < ridge energy
Ev = model(jnp.array([[-1.0],[1.0]])); Er = model(jnp.array([[0.0]]))
print(f"E_valley = {np.array(Ev).ravel()},  E_ridge = {float(Er[0]):.3f}")
assert Ev.mean() < Er[0], "valleys should be lower-energy than the ridge"
print("PASS: E_valley < E_ridge  -> the network learned the double-well shape.")


## 6. Sampling = Brownian motion in $E_\theta$  ·  ⭐ Exercise (TODO 3)

Having an energy is only half the story; we want **samples**. The physics answer is **overdamped Langevin dynamics** — a particle diffusing in the potential $E_\theta$:

$$
dx = -\nabla_x E_\theta(x)\,dt + \sqrt{2}\,dW .
$$

Its stationary distribution is exactly $p_\theta \propto e^{-E_\theta}$ (this is the fluctuation–dissipation balance: the deterministic drift down the energy gradient is balanced by thermal noise). Discretize with step $\eta$:

$$
\boxed{\;x_{k+1} = x_k \;-\; \tfrac{\eta}{2}\,\nabla_x E_\theta(x_k) \;+\; \sqrt{\eta}\,\xi_k,\qquad \xi_k\sim\mathcal N(0,I)\;}
$$

This is **unadjusted Langevin** (no Metropolis correction). The drift $-\tfrac{\eta}{2}\nabla E = +\tfrac{\eta}{2} s_\theta$ pulls toward low energy; the noise $\sqrt\eta\,\xi$ provides thermal kicks. It *is* Brownian motion in a force field — overdamped, like a colloid in water.

> ### ⭐ TODO 3 — complete one Langevin step
> **Verification:** run a 10,000-step chain; its histogram should match the training data (we'll quantify with a KL divergence, target $< 0.3$).

In [ ]:
def langevin_step(model, x, key, eta=1e-2):
    '''One unadjusted Langevin step. x: (n_chains, d) -> (n_chains, d).'''
    xi = jr.normal(key, x.shape)
    # ---------------------------------------------------------------
    # TODO 3:  x_new = x - (eta/2) * grad_x E(x) + sqrt(eta) * xi
    #   grad_x E(x) = -score_fn(model, x)
    grad_E = -score_fn(model, x)
    x_new = x - 0.5 * eta * grad_E + jnp.sqrt(eta) * xi
    return x_new

@partial(nnx.jit, static_argnums=(3, 4))
def langevin_chain(model, x0, key, n_steps, eta):
    def body(carry, k):
        x = carry
        x = langevin_step(model, x, k, eta)
        return x, x
    keys = jr.split(key, n_steps)
    xf, traj = jax.lax.scan(body, x0, keys)
    return xf, traj      # xf: (n_chains, d); traj: (n_steps, n_chains, d)

# 200 chains, 10k steps, started from the standard normal
n_chains, n_steps = 200, 10000
x0 = jr.normal(jr.PRNGKey(0), (n_chains, 1))
_, traj = langevin_chain(model, x0, jr.PRNGKey(1), n_steps, 2e-2)
samples = traj[2000:].reshape(-1, 1)     # discard 2000-step burn-in
print(f"trajectory shape: {traj.shape}")
print(f"samples (post burn-in) shape: {samples.shape}")


In [ ]:
# Compare Langevin samples to true Boltzmann density; quantify with KL.
xs_p = jnp.linspace(-2.5, 2.5, 200)
p_true = jnp.exp(-U_double_well(xs_p)); p_true = p_true / jnp.trapezoid(p_true, xs_p)

hist, edges = np.histogram(np.array(samples[:,0]), bins=xs_p, density=True)
centers = 0.5 * (edges[:-1] + edges[1:])
# discrete KL(data_hist || p_true) on overlapping bins
pt = np.interp(centers, np.array(xs_p), np.array(p_true))
eps = 1e-8
kl = float(np.sum(hist * np.log((hist + eps) / (pt + eps))) * np.diff(edges).mean())

plt.figure(figsize=(5.5, 3.2))
plt.plot(xs_p, p_true, lw=2, label="true Boltzmann")
plt.hist(np.array(samples[:,0]), bins=80, density=True, alpha=0.5, label="Langevin samples")
plt.title(f"Langevin EBM samples   (KL ≈ {kl:.3f})"); plt.legend(); plt.xlabel("x")
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

print(f"KL(samples || true) ≈ {kl:.3f}   (checkpoint target: < 0.3)")


## 7. ⭐ CHECKPOINT B — slow mixing across barriers

Here is where the physics bites back. Langevin dynamics must **cross the energy barrier** at $x=0$ to move between valleys. If the barrier is high relative to the noise, a short chain gets **trapped**. This is *not* a bug in our code — it is the same ergodicity problem that plagues real molecular dynamics and Monte Carlo, and it is the Achilles heel of every EBM sampler.

We compare three runs and look at the **occupation ratio** $r = \#\{x>0\} / N$ (fraction of time in the right valley). The training baseline is $r \approx 0.5$ (symmetric well).

In [ ]:
def occupation_right(chain_samples):
    return float(np.mean(np.array(chain_samples[:, 0]) > 0.0))

# (a) start in LEFT valley, short chain
x0_left = -1.0 * jnp.ones((200, 1))
_, ta = langevin_chain(model, x0_left, jr.PRNGKey(10), 1000, 2e-2)
# (b) start in RIGHT valley, short chain
x0_right = 1.0 * jnp.ones((200, 1))
_, tb = langevin_chain(model, x0_right, jr.PRNGKey(11), 1000, 2e-2)
# (c) long chain
x0_c = jr.normal(jr.PRNGKey(12), (200, 1))
_, tc = langevin_chain(model, x0_c, jr.PRNGKey(13), 50000, 2e-2)

ra = occupation_right(ta.reshape(-1, 1))
rb = occupation_right(tb.reshape(-1, 1))
rc = occupation_right(tc[5000:].reshape(-1, 1))

print(f"(a) start LEFT,  1k steps : right-valley occupation r = {ra:.3f}")
print(f"(b) start RIGHT, 1k steps : right-valley occupation r = {rb:.3f}")
print(f"(c) long chain, 50k steps : right-valley occupation r = {rc:.3f}")
print(f"training baseline         : r ≈ 0.5 (symmetric double-well)")
print()
print("Interpretation:")
print(" - (a) stays near r≈0  : trapped in the LEFT valley, never crossed.")
print(" - (b) stays near r≈1  : trapped in the RIGHT valley.")
print(" - (c) r≈0.5           : long enough to cross the barrier many times -> mixed.")
print("=> EBM samples are only as good as the sampler's ability to cross barriers.")


In [ ]:
# Visualize the three chains' trajectories (first chain of each)
fig, ax = plt.subplots(1, 3, figsize=(12, 3), sharey=True)
ax[0].plot(np.array(ta[:, 0, 0])); ax[0].set_title("(a) start LEFT, 1k"); ax[0].axhline(0, color="k", lw=0.7)
ax[1].plot(np.array(tb[:, 0, 0])); ax[1].set_title("(b) start RIGHT, 1k"); ax[1].axhline(0, color="k", lw=0.7)
ax[2].plot(np.array(tc[:5000, 0, 0])); ax[2].set_title("(c) long chain (first 5k)"); ax[2].axhline(0, color="k", lw=0.7)
for a_ in ax: a_.set_xlabel("step"); a_.grid(alpha=0.3)
ax[0].set_ylabel("x position")
plt.tight_layout(); plt.show()


## 8. ⭐ EXTENSION — DSM secretly learns an *implicit* energy

We claimed two things were the same object: the **trained score** $s_\theta$ and the **negative gradient of the energy** $-\nabla E_\theta$. By construction in *our* model they are literally equal (we built $s = -\nabla E$). But the deeper statement is **Vincent's theorem**: the minimizer of the DSM loss is the score of the *smoothed data density* $p_\sigma$,

$$
s_{\theta^\*}(\tilde x) \;=\; \nabla_{\tilde x}\log p_\sigma(\tilde x).
$$

So even a network that outputs only a *vector* score (as in DDPM, with **no explicit energy**) is implicitly modeling $-\nabla E_{\text{eff}}$ for the effective potential $E_{\text{eff}} = -\log p_\sigma$. The EBM just makes that energy explicit and guarantees the field is conservative.

Let's verify numerically that our learned score equals the **analytic score of the noised data density**, computed *independently* by differentiating a kernel-density estimate of $\log p_\sigma$.

In [ ]:
# Independent ground truth: log p_sigma(x) from the noised TRUE Boltzmann density.
# p_sigma = p_data * Gaussian(0, sigma^2)  (convolution). We get it by KDE on noised data.
key_n, _ = jr.split(jr.PRNGKey(99))
X_noised = X + SIGMA * jr.normal(key_n, X.shape)      # (N,1) data convolved with kernel

def log_p_sigma(x_scalar, bw=0.12):
    # Gaussian KDE log-density of the noised data at scalar x
    diffs = (x_scalar - X_noised[:, 0]) / bw
    logk = -0.5 * diffs**2 - jnp.log(bw) - 0.5*jnp.log(2*jnp.pi)
    return jax.scipy.special.logsumexp(logk) - jnp.log(X_noised.shape[0])

# analytic score of the smoothed density: d/dx log p_sigma
score_kde = jax.vmap(jax.grad(log_p_sigma))
xg = jnp.linspace(-2.0, 2.0, 200)
s_kde = score_kde(xg)
s_model = score_fn(model, xg[:, None])[:, 0]

plt.figure(figsize=(5.8, 3.4))
plt.plot(xg, s_kde, lw=2, label=r"$\nabla\log p_\sigma$ (KDE of noised data)")
plt.plot(xg, s_model, "--", lw=2, label=r"$-\nabla E_\theta$ (learned EBM)")
plt.title("DSM minimizer = score of smoothed density"); plt.legend()
plt.xlabel("x"); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

corr = float(np.corrcoef(np.array(s_kde), np.array(s_model))[0, 1])
print(f"correlation between the two scores: {corr:.3f}  (close to 1 => same object)")


**The closed loop.** L17 said *DDPM $\epsilon$-prediction = score regression*. L18 says *score = $-\nabla E_\theta$ of an energy-based model*. The plot above shows both equal $\nabla\log p_\sigma$ — **the same physical force field**, reached from diffusion and from statistical mechanics. The two halves of the generative-models block are one idea.

## 9. Real data — the alanine-dipeptide free-energy surface

Time for a real molecule. **Alanine dipeptide** (Ac-Ala-NMe) is the *Drosophila* of biomolecular simulation: its slow conformational dynamics are captured by two backbone dihedral angles, $\phi$ and $\psi$ (the **Ramachandran** coordinates). The 2D free-energy surface

$$
F(\phi,\psi) = -k_BT\,\log p(\phi,\psi)
$$

has several metastable basins — notably **C7eq** (≈ $(-80°, +80°)$) and **C7ax** / $\alpha_R$ regions — separated by barriers of a few $k_BT$. We will train a 2D EBM directly on $(\phi,\psi)$ samples, exactly as in 1D, and recover this surface. This is the **same target molecule we sampled with flows (L15), diffusion (L16/17)** — the data is shared across the whole generative block.

**Two ways to get the data** (the next cell tries them in order):
1. Download a precomputed `(phi, psi)` array of ~10k frames (a slice of a public 300 K AMBER ff99SB-ILDN trajectory).
2. If that is unavailable, extract dihedrals from a short MD trajectory with `mdtraj`.
3. Fallback: a faithful **synthetic** Ramachandran mixture so the notebook always runs offline.

In [ ]:
# ---- Load alanine-dipeptide (phi, psi) in DEGREES, shape (N, 2) ----
# We try a real download first, then mdtraj, then a synthetic fallback so the
# notebook ALWAYS runs (Colab CPU, offline-safe).
import os, urllib.request

def load_alanine_phi_psi():
    # Option 1: precomputed dihedrals (uncomment + set a working URL on Colab)
    # url = "https://<host>/alanine_dipeptide_phi_psi.npy"   # (N,2) degrees
    # path = "/tmp/ala_phipsi.npy"
    # urllib.request.urlretrieve(url, path)
    # return np.load(path)
    raise RuntimeError("no precomputed file configured")

def load_alanine_mdtraj():
    # Option 2: extract from a trajectory with mdtraj (needs traj.xtc + top.pdb).
    import mdtraj as md
    traj = md.load("alanine.xtc", top="alanine.pdb")     # provide these on Colab
    phi = md.compute_phi(traj)[1][:, 0]
    psi = md.compute_psi(traj)[1][:, 0]
    return np.rad2deg(np.stack([phi, psi], axis=1))

def synthetic_ramachandran(n=10000, seed=0):
    '''Faithful offline stand-in: Gaussian mixture over known basins (degrees).'''
    rng = np.random.default_rng(seed)
    # (mean_phi, mean_psi, weight) for C7eq, C5, alpha_R, C7ax
    basins = [(-80,  80, 0.45),   # C7eq / beta
              (-150, 160, 0.20),  # C5 / extended
              (-70, -40, 0.25),   # alpha_R (right-handed helix)
              ( 60, -80, 0.10)]   # C7ax / alpha_L (rare)
    comps = rng.choice(len(basins), size=n, p=[b[2] for b in basins])
    out = np.empty((n, 2))
    for i, (mp, ms, _) in enumerate(basins):
        m = comps == i
        out[m, 0] = rng.normal(mp, 15, m.sum())
        out[m, 1] = rng.normal(ms, 15, m.sum())
    # wrap to [-180, 180)
    return ((out + 180) % 360) - 180

try:
    PHIPSI = load_alanine_phi_psi(); src = "precomputed download"
except Exception:
    try:
        PHIPSI = load_alanine_mdtraj(); src = "mdtraj trajectory"
    except Exception:
        PHIPSI = synthetic_ramachandran(); src = "synthetic fallback"

print(f"alanine (phi,psi) source: {src}")
print(f"PHIPSI shape: {PHIPSI.shape}   units: degrees, range ~[-180,180]")


In [ ]:
# Ramachandran scatter + 2D histogram (the empirical free-energy surface)
fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
ax[0].scatter(PHIPSI[:,0], PHIPSI[:,1], s=3, alpha=0.25)
ax[0].set_xlim(-180,180); ax[0].set_ylim(-180,180)
ax[0].set_xlabel(r"$\phi$ (deg)"); ax[0].set_ylabel(r"$\psi$ (deg)"); ax[0].set_title("Ramachandran scatter")

H, xe, ye = np.histogram2d(PHIPSI[:,0], PHIPSI[:,1], bins=60, range=[[-180,180],[-180,180]], density=True)
F = -np.log(H.T + 1e-6)      # free energy in kT units; -log p
F -= F.min()
im = ax[1].imshow(F, origin="lower", extent=[-180,180,-180,180], aspect="auto", cmap="viridis_r", vmax=8)
ax[1].set_xlabel(r"$\phi$ (deg)"); ax[1].set_ylabel(r"$\psi$ (deg)"); ax[1].set_title(r"empirical $F=-\log p$  (kT)")
plt.colorbar(im, ax=ax[1], label="F / kT")
plt.tight_layout(); plt.show()


### 9.1 Train a 2D EBM on $(\phi,\psi)$

We rescale angles to roughly unit variance (degrees → /180) so the network and Langevin steps behave numerically, train with the **same** DSM loss, then visualize the learned energy surface — **CHECKPOINT A**.

In [ ]:
# Normalize angles to ~[-1,1] for stable training; remember the scale.
ANG = 180.0
Xa = jnp.asarray(PHIPSI / ANG)                # (N, 2) in ~[-1,1]
print(f"2D training data shape: {Xa.shape}")

model2d = EnergyNet(d_in=2, hidden=128, rngs=nnx.Rngs(0))
opt2d = nnx.Optimizer(model2d, optax.adam(2e-3), wrt=nnx.Param)

SIGMA2 = 0.07
loss_hist2 = []
tkey = jr.PRNGKey(7)
for it in range(2500):
    tkey, bk, nk = jr.split(tkey, 3)
    idx = jr.randint(bk, (512,), 0, Xa.shape[0])
    loss = train_step(model2d, opt2d, Xa[idx], nk, SIGMA2)
    loss_hist2.append(float(loss))
    if it % 500 == 0:
        print(f"step {it:4d}   DSM loss = {float(loss):.4f}")

plt.figure(figsize=(5,3)); plt.plot(loss_hist2); plt.yscale("log")
plt.xlabel("step"); plt.ylabel("DSM loss"); plt.title("2D EBM training"); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
# ⭐ CHECKPOINT A: learned energy landscape E_theta(phi, psi) as a contour map.
gp = jnp.linspace(-1, 1, 120)
PHI, PSI = jnp.meshgrid(gp, gp)
grid = jnp.stack([PHI.ravel(), PSI.ravel()], axis=1)     # (120*120, 2)
Eg = np.array(model2d(grid)).reshape(PHI.shape)
Eg -= Eg.min()

plt.figure(figsize=(5.6, 4.6))
cs = plt.contourf(PHI*ANG, PSI*ANG, Eg, levels=25, cmap="viridis_r")
plt.colorbar(cs, label=r"$E_\theta$ (kT, shifted)")
plt.xlabel(r"$\phi$ (deg)"); plt.ylabel(r"$\psi$ (deg)")
plt.title(r"CHECKPOINT A: learned $E_\theta(\phi,\psi)$")
# mark the canonical basins for visual comparison
for (mp, ms, name) in [(-80,80,"C7eq"), (-70,-40,r"$\alpha_R$"), (60,-80,"C7ax")]:
    plt.scatter([mp],[ms], c="white", edgecolors="k", s=60)
    plt.annotate(name, (mp, ms), textcoords="offset points", xytext=(6,6), color="white")
plt.tight_layout(); plt.show()
print("Visual check: energy minima (dark) should sit on the C7eq / alpha_R basins.")


In [ ]:
# Sample the 2D EBM with Langevin and overlay on the data (post-training payoff).
x0_2d = jr.normal(jr.PRNGKey(0), (500, 2)) * 0.3
_, traj2d = langevin_chain(model2d, x0_2d, jr.PRNGKey(1), 8000, 5e-3)
samp2d = np.array(traj2d[2000:].reshape(-1, 2)) * ANG     # back to degrees

fig, ax = plt.subplots(1, 2, figsize=(11, 4.2), sharex=True, sharey=True)
ax[0].hist2d(PHIPSI[:,0], PHIPSI[:,1], bins=60, range=[[-180,180],[-180,180]], cmap="Blues")
ax[0].set_title("training data"); ax[0].set_xlabel(r"$\phi$"); ax[0].set_ylabel(r"$\psi$")
ax[1].hist2d(samp2d[:,0], samp2d[:,1], bins=60, range=[[-180,180],[-180,180]], cmap="Oranges")
ax[1].set_title("EBM Langevin samples"); ax[1].set_xlabel(r"$\phi$")
plt.tight_layout(); plt.show()
print("If basins line up, the EBM learned the conformational free-energy surface.")


## 10. EBM vs. the L15 Boltzmann-generator baseline

Where does the EBM sit relative to the flows of L15? Both sample $p\propto e^{-E}$, but they make opposite trade-offs:

| | **Normalizing Flow / Boltzmann generator (L15)** | **Energy-Based Model (L18)** |
|---|---|---|
| samples | **one** forward pass, i.i.d. | **long** Langevin chain, correlated |
| exact $\log p$? | yes (change of variables) | no (needs $Z$) |
| architecture | constrained to be **invertible** | **any** scalar net — unconstrained |
| reweighting to target | yes (importance weights w/ exact $\log p$) | hard (no $\log p$) |
| failure mode | mode collapse if flow too weak | **slow mixing** across barriers (§7) |
| training | maximum likelihood / reverse-KL | score matching (no $Z$) |

The honest summary for a physicist: **flows give you cheap i.i.d. samples and a tractable density but pay with an architectural straitjacket; EBMs give you an unconstrained energy but pay with MCMC and its ergodicity problems.** Diffusion models (L17) are, in this light, EBMs whose multi-noise-level score makes the sampling chain mix far better — which is exactly why they won at scale.

In [ ]:
# Tiny side-by-side: number of model evaluations to get N "independent" samples.
N = 1000
flow_evals = N                 # one decoder pass each (idealized)
# EBM: autocorrelation time tau steps per independent sample (rough, from §7 mixing)
tau = 200
ebm_evals = N * tau
print(f"To obtain ~{N} independent samples:")
print(f"  flow / Boltzmann generator : ~{flow_evals:>8,d} energy/decoder evaluations")
print(f"  EBM via Langevin (tau~{tau}): ~{ebm_evals:>8,d} energy-gradient evaluations")
print("=> EBMs trade architectural freedom for sampling cost (the MCMC tax).")


## 11. Forward pointer → L19: simulation-based inference

The score you built today reappears one more time. In **L19 (Simulation-Based Inference)** we will want the *posterior* $p(\vartheta \mid x)$ over physical parameters $\vartheta$ given an observation $x$. Two families:

- **Neural Posterior Estimation (NPE):** directly fit a density $q_\phi(\vartheta\mid x)$ (a conditional flow, à la L15).
- **Score-based / diffusion posterior estimation:** learn the *posterior score* $\nabla_\vartheta \log p(\vartheta\mid x)$ — *exactly today's object*, now conditioned on data — and Langevin-sample the posterior.

So the EBM/score viewpoint of L18 is not a dead end: it is the bridge from generative modeling to **inference**, where "energy" becomes "negative log-posterior" and Langevin sampling becomes posterior sampling. We close the generative block here and pick up that thread next time.

## 12. Summary & take-home exercises

**What you built, in one breath:** a neural energy $E_\theta$ → score $s_\theta=-\nabla E_\theta$ by autograd → trained it with denoising score matching (no $Z$!) → sampled it with Langevin = Brownian motion → watched it get stuck at barriers → applied the whole pipeline to the alanine-dipeptide free-energy surface.

**The three identities to remember:**
1. $E(x) = -\log p(x) + \text{const}$, and the const is $-\log Z$ — intractable, and *that's why we use scores*.
2. $s_\theta = \nabla\log p_\theta = -\nabla E_\theta$ — the $Z$ drops out; this *is* the L17 diffusion score.
3. Langevin $x_{k+1}=x_k-\tfrac\eta2\nabla E_\theta+\sqrt\eta\,\xi$ is overdamped Brownian dynamics with stationary law $e^{-E_\theta}$.

**Exercises (the ⭐ TODOs + checkpoints, recapped):**

- **TODO 1** — `EnergyNet`: scalar output `(B,)`, smooth activations; verify $E_{\text{valley}}<E_{\text{ridge}}$. *(done in §3/§5.2)*
- **TODO 2** — `dsm_loss`: noise with $\sigma$, target $-\epsilon/\sigma$; verify loss positive & decreasing. *(done in §5)*
- **TODO 3** — `langevin_step`; run a 10k chain, KL to truth $<0.3$. *(done in §6)*
- **CHECKPOINT A** — contour the learned 2D energy; minima on C7eq / $\alpha_R$. *(done in §9)*
- **CHECKPOINT B** — occupation ratios for short-left, short-right, long chains; observe trapping. *(done in §7)*
- **EXTENSION** — show numerically $-\nabla E_\theta = \nabla\log p_\sigma$. *(done in §8)*

**Stretch goals (try on your own):**
1. **Annealed / multi-noise DSM.** Train with several $\sigma$ levels and run *annealed* Langevin (large $\sigma$ → small $\sigma$). Does mixing across the double-well barrier improve? (This is the L17 idea retrofitted onto the EBM — and it should fix §7's trapping.)
2. **Persistent Contrastive Divergence.** Implement the maximum-likelihood gradient $\nabla_\theta \mathcal L = \mathbb E_{\text{data}}[\nabla_\theta E_\theta] - \mathbb E_{p_\theta}[\nabla_\theta E_\theta]$ with a persistent Langevin sampler for the second term. Compare to DSM.
3. **Temperature control.** Add a temperature $T$ by sampling $p\propto e^{-E_\theta/T}$ (scale the noise by $\sqrt T$). Reproduce a free-energy vs. $T$ curve for the double-well.
4. **Real dihedrals.** Wire up `mdtraj` (or download a real trajectory) to replace the synthetic Ramachandran data and compare the learned $F(\phi,\psi)$ to a published surface.


## 13. References

**Energy-based models & score matching**

1. Hyvärinen, *Estimation of Non-Normalized Statistical Models by Score Matching*, JMLR 6 (2005).
2. Vincent, *A Connection Between Score Matching and Denoising Autoencoders*, Neural Computation 23 (2011).
3. Song & Ermon, *Generative Modeling by Estimating Gradients of the Data Distribution*, NeurIPS 2019. arXiv:1907.05600.
4. Song *et al.*, *Score-Based Generative Modeling through Stochastic Differential Equations*, ICLR 2021. arXiv:2011.13456. *(the L17↔L18 SDE/score bridge)*
5. Du & Mordatch, *Implicit Generation and Modeling with Energy-Based Models*, NeurIPS 2019. arXiv:1903.08689.

**Physics & sampling**

6. Noé, Olsson, Köhler & Wu, *Boltzmann Generators*, *Science* 365, eaaw1147 (2019). *(L15 baseline)*
7. Parisi, *Correlation functions and computer simulations*, Nucl. Phys. B 180 (1981). *(Langevin sampling in field theory)*
8. Roberts & Tweedie, *Exponential Convergence of Langevin Distributions and Their Discrete Approximations*, Bernoulli 2 (1996). *(unadjusted Langevin theory)*

**Background**

9. LeCun *et al.*, *A Tutorial on Energy-Based Learning* (2006).
10. Cranmer, Brehmer & Louppe, *The frontier of simulation-based inference*, PNAS 117 (2020). *(L19 forward pointer)*

---

*End of Lecture 18 — and of the generative-models block. Next: simulation-based inference.*
